# 21. Kiểm tra tái lập (GĐ J, task 9.3), CPU
Clone repo sạch tại commit hiện tại của `main`, chạy lại `scripts/aggregate_h.py` từ các output Kaggle đã lưu (17 test, 17a val, 04 và 11 đồ thị),
rồi so `results.csv`, `summary_test.csv`, `summary_val.csv`, `summary_test_tiers.csv` với bản trong repo (lệch F1-macro ≤ 0,5 điểm %).
Không huấn luyện, không chạy lại test.

In [ ]:
import glob, json, os, resource, subprocess, sys, time
from contextlib import contextmanager
from pathlib import Path

WORK = "/kaggle/working"
REPO = "/tmp/gnn-iot-ids"
URL = "https://github.com/minucedev/gnn-iot-ids.git"
INFO = {"steps_s": {}, "device": "CPU", "gpu": None}
T_START = time.time()


def cpu_model():
    try:
        for line in open("/proc/cpuinfo"):
            if line.startswith("model name"):
                return line.split(":", 1)[1].strip()
    except OSError:
        pass
    return "không rõ"


def peak_ram_gb():
    kb = max(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss, resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss)
    return round(kb / 2**20, 2)


def save_info():
    INFO["total_s"] = round(time.time() - T_START, 1)
    INFO["peak_ram_gb"] = peak_ram_gb()
    INFO["gpu_hours"] = 0
    with open(f"{WORK}/run_info.json", "w", encoding="utf-8") as f:
        json.dump(INFO, f, ensure_ascii=False, indent=1)


@contextmanager
def step(name):
    t0 = time.time()
    try:
        yield
    finally:
        INFO["steps_s"][name] = round(time.time() - t0, 1)
        save_info()
        print(f"--- {name}: {INFO['steps_s'][name]} s, RAM đỉnh tới nay {INFO['peak_ram_gb']} GB")


def run(cmd, ok=(0,), log=None, **kw):
    if log:
        with open(log, "w", encoding="utf-8") as f:
            r = subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT, text=True, **kw)
        print(open(log, encoding="utf-8").read()[-6000:])
    else:
        r = subprocess.run(cmd, capture_output=True, text=True, **kw)
        print(r.stdout[-9000:], end="")
        print(r.stderr[-2500:], end="", file=sys.stderr)
    print(f"[mã thoát {r.returncode}]")
    if r.returncode not in ok:
        raise RuntimeError(f"Lệnh lỗi: {' '.join(cmd)}")
    return r


def find(pattern):
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    assert hits, f"Không thấy {pattern} trong /kaggle/input (đã gắn đủ nguồn dữ liệu chưa?)"
    return hits[0]


with step("clone_va_cai_goi"):
    run(["rm", "-rf", REPO])
    run(["git", "clone", "--depth", "1", URL, REPO])
    INFO["commit"] = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
    with open(f"{REPO}/requirements.txt") as f:
        reqs = [l.strip() for l in f if l.strip() and not l.startswith("#")]
    keep = [r for r in reqs if r.split("==")[0].lower() not in {"torch", "jupyter", "ipykernel"}]
    run([sys.executable, "-m", "pip", "install", "-q", *keep])
INFO["cpu"] = cpu_model()
INFO["cpu_count"] = os.cpu_count()
save_info()
print("Commit:", INFO["commit"], "| CPU:", INFO["cpu"])

## pytest cho phần tổng hợp và so sánh (dừng nếu lỗi)

In [ ]:
with step("pytest"):
    r = run([sys.executable, "-m", "pytest", "-q", "tests/test_aggregate_h.py", "tests/test_reproduce_check.py"], cwd=REPO)
    INFO["pytest"] = r.stdout.strip().splitlines()[-1]
save_info()

## Đường dẫn đầu vào (như notebook 18)

In [ ]:
EVAL_SPLIT, EVAL_SLUG = "test", "gnn-iot-ids-17-final-test"
DS_T, DS_B = "nf_ton_iot_v3", "nf_bot_iot_v3"
EVAL_DIR = os.path.dirname(find(f"{EVAL_SLUG}/results_{EVAL_SPLIT}_{DS_T}.csv"))
VAL_DIR = os.path.dirname(find(f"gnn-iot-ids-17a-dryrun-val/results_val_{DS_T}.csv"))
REGEN = f"{WORK}/regen"
CFG = dict(eval_split=EVAL_SPLIT, eval_dir=EVAL_DIR, val_dir=VAL_DIR, repo=REPO, out=REGEN,
           data={DS_T: os.path.dirname(find(f"{DS_T}/split.json")), DS_B: os.path.dirname(find(f"{DS_B}/split.json"))})
os.makedirs(REGEN, exist_ok=True)
json.dump(CFG, open(f"{WORK}/aggregate_config.json", "w"), ensure_ascii=False, indent=1)
INFO["config"] = CFG
save_info()

## Chạy lại tổng hợp rồi so với bảng trong repo

In [ ]:
with step("aggregate_h"):
    run([sys.executable, "-I", "scripts/aggregate_h.py", f"{WORK}/aggregate_config.json"], log=f"{WORK}/aggregate_h_log.txt", cwd=REPO)
with step("reproduce_check"):
    r = run([sys.executable, "-I", "scripts/reproduce_check.py", REGEN, f"{REPO}/reports/tables", f"{WORK}/reproduce_check.json"], ok=(0, 3), cwd=REPO)
res = json.load(open(f"{WORK}/reproduce_check.json", encoding="utf-8"))
INFO["reproduce"] = {"passed": res["passed"], "max_abs_diff_f1_macro": res["max_abs_diff_f1_macro"],
                     "max_abs_diff_any_numeric_column": res["max_abs_diff_any_numeric_column"],
                     "tables": {k: {"rows_new": v["rows_new"], "rows_ref": v["rows_ref"], "same_rows": v["same_rows"],
                                    "max_abs_diff_f1_macro": v["max_abs_diff_f1_macro"]} for k, v in res["tables"].items()}}
save_info()
# giữ lại bản bảng chính đã chạy lại (nhỏ) để đối chiếu, bỏ hình và file lớn
import shutil
for fn in os.listdir(REGEN):
    if fn.endswith(".png") or fn.startswith("pred_"):
        os.remove(f"{REGEN}/{fn}")
print(json.dumps(INFO["reproduce"], indent=1, ensure_ascii=False))
print(os.listdir(WORK), os.listdir(REGEN))